# Nimbus Weather Station - Setup & Requirements

## System Architecture

```
[ Raspberry Pi ]                [ Fog Node / PC ]
  sensors.py                     docker-compose.yml
  NimbusServer.py  ──MQTT──►  RabbitMQ (broker)
  Flask API :5000                    │
                              ┌──────┴──────┐
                      Data_Modification.py  DataPublisher.py
                              └──────┬──────┘
                                 InfluxDB :8086
```

| Component | Runs on | Purpose |
|---|---|---|
| `sensors.py` | Raspberry Pi | Reads temperature, humidity, pressure from SenseHat |
| `NimbusServer.py` | Raspberry Pi | Flask API + publishes raw sensor data to MQTT |
| `Data_Modification.py` | Fog Node / PC | Subscribes to raw data, computes moving averages & daily extremes, writes to InfluxDB |
| `DataPublisher.py` | Fog Node / PC | Subscribes to raw data, writes raw values to InfluxDB |
| `docker-compose.yml` | Fog Node / PC | Runs RabbitMQ (MQTT broker) and InfluxDB |

---

## Network & Access

| | Value |
|---|---|
| Raspberry Pi IP | `192.168.32.11` |
| SSH user | `admin` |
| SSH password | `admin` |
| Fog Node / Broker IP | `192.168.32.6` |
| RabbitMQ Management UI | `http://192.168.32.6:15672` |
| InfluxDB UI | `http://192.168.32.6:8086` |
| Flask Dashboard | `http://192.168.32.11:5000` |

---
## Default Credentials

| Service | Username | Password |
|---|---|---|
| RabbitMQ | `Nimbus` | `Nimbus` |
| InfluxDB | `Nimbus` | `NimbusWeather26` |

InfluxDB org: `Nimbus` - Bucket: `Nimbus`

InfluxDB token (set in docker-compose and all scripts):
```
xYitY0VJ5e5hyCt6uR-fhA2NiCNlxbwl_QAj8_Xj-VIiy5VsRWauLodSQMtLLYr1ANnxXrwNSH_Tz0jBQMpqjQ==
```
> ⚠️ If you re-initialise InfluxDB, generate a new token via the UI and update it in `NimbusServer.py`, `Data_Modification.py`, and `DataPublisher.py`.

---
# Part 1 - Fog Node / PC Setup

## Step 1 - Install Docker Desktop
Download and install Docker Desktop from https://www.docker.com/products/docker-desktop/

## Step 2 - Start RabbitMQ and InfluxDB
Open a terminal, navigate to the project root (where `docker-compose.yml` lives), and run:

In [ ]:
docker-compose up --build

Wait until both containers report healthy. You can verify:
- RabbitMQ: http://192.168.32.6:15672 (login: Nimbus / Nimbus)
- InfluxDB: http://192.168.32.6:8086 (login: Nimbus / NimbusWeather26)

## Step 3 - Create Python Environment (Fog Node)
The fog-node scripts (`Data_Modification.py`, `DataPublisher.py`) require Python 3.10.

In [ ]:
conda create --name Nimbus_env python=3.10
conda activate Nimbus_env

## Step 4 - Install Fog Node Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install paho-mqtt influxdb-client

## Step 5 - Run the Fog Node Services
Open **two separate terminals** (both with `Nimbus_env` activated) and run one script in each:

In [ ]:
# Terminal 1 - moving averages, daily extremes, writes processed data to InfluxDB
python Data_Modification.py

# Terminal 2 - forwards raw sensor readings to InfluxDB
python DataPublisher.py

Expected output from `Data_Modification.py`:
```
Connected to broker - subscribing now
Subscription confirmed at QoS 1
Data analytics are running and listening for data
```

---
# Part 2 - Raspberry Pi Setup

## Step 1 - SSH into the Raspberry Pi

In [ ]:
ssh admin@192.168.32.11

## Step 2 - Clone the Repository

In [ ]:
git clone <your-repo-url>
cd IoT_Group8

## Step 3 - Install Raspberry Pi Dependencies

`sense-hat` is pre-installed on Raspberry Pi OS. Install the remaining packages:

In [ ]:
import sys
!{sys.executable} -m pip install paho-mqtt influxdb-client flask sense-hat

## Step 4 - Run NimbusServer

Make sure the Fog Node Docker services are already running before starting this.

In [ ]:
python NimbusServer.py

Expected output:
```
[QoS 2] Subscribed to config topic ✓
[QoS 2] Config subscription confirmed ✓
Connected to MQTT broker at 192.168.32.6
 * Running on http://0.0.0.0:5000
```

The Flask dashboard is then available at **http://192.168.32.11:5000**

---
# Part 3 - REST API Reference

All endpoints are served by `NimbusServer.py` on port `5000`.

| Method | Endpoint | Description |
|---|---|---|
| GET | `/` | HTML dashboard (live temperature, humidity, pressure + alert status) |
| GET | `/api/data` | Current sensor readings + active alert states (JSON) |
| GET | `/api/data/history?limit=N` | Last N readings from InfluxDB (default 50) |
| POST | `/api/alerts` | Create a new threshold alert |
| DELETE | `/api/alerts/<id>` | Delete an alert by ID |
| PUT | `/api/config` | Update publish interval or temperature offset (delivered via MQTT QoS 2) |

### Example - Create an alert

In [ ]:
curl -X POST http://192.168.32.11:5000/api/alerts \
     -H "Content-Type: application/json" \
     -d '{"sensor": "temperature", "condition": "above", "threshold": 30}'

### Example - Update publish interval and temperature offset

In [ ]:
curl -X PUT http://192.168.32.11:5000/api/config \
     -H "Content-Type: application/json" \
     -d '{"publish_interval": 5, "temp_offset": -2.5}'

---
# Part 4 - MQTT Topics

| Topic | Publisher | Subscriber | QoS | Payload |
|---|---|---|---|---|
| `nimbus/sensor_data` | `NimbusServer.py` | `Data_Modification.py`, `DataPublisher.py` | 0 | `{temperature, humidity, pressure}` |
| `nimbus/processed_data` | `Data_Modification.py` | - | 1 | `{avg_temperature, avg_humidity, avg_pressure, max_temp, min_temp}` |
| `nimbus/config/rpi2` | `NimbusServer.py` (via PUT /api/config) | `NimbusServer.py` config listener | 2 | `{publish_interval?, temp_offset?}` |